# Notebook 02 — Baseline Experiments with SageMaker MLflow App

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** B — Model Development + C — MLOps (Experiment Tracking)  
**Estimated Runtime:** 5-10 minutes

---

## What this notebook does

This is team03's adaptation of the course's Notebook 02, using the **SageMaker MLflow App** created in `01A_setup_sagemaker_mlflow_app_team03.ipynb`.

1. Connects to team03's SageMaker MLflow App and validates the `TeamId` tag
2. Logistic Regression baseline — logs params, metrics, plots, and model artifacts to MLflow
3. Random Forest baseline — logs params, metrics, feature importance, and model artifacts
4. Hyperparameter tuning with nested child runs
5. Programmatic run comparison and best-model selection against a quality gate
6. Saves `best_model.json` for hand-off to Notebook 03

## Expected setup from Notebook 01A

Notebook 01A should have created team03's MLflow App and saved a local config file:

```text
mlflow_app_config_team03_s301.json
```

Team03's App (from `01A_setup_sagemaker_mlflow_app_team03.ipynb`):

```text
arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-J5AYUG4AJHVW
```

This notebook also validates that the selected MLflow App is tagged with the correct `TeamId` before logging.

> **Adapted for team03 (Ong Hui Lin, Student 2 — MLOps & Deployment) from the ITI113 course template notebook `02_baseline_experiments_sagemaker_mlflow_app_with_(team_tag_check).ipynb`.** The MLflow connection, team-tag validation, model training, quality gate, and best-model hand-off logic are unchanged from the tutor's original. What had to change, because this project is text classification (TF-IDF) rather than tabular data, is marked **`# ADAPTED`** below: how data is loaded (the tutor's Notebook 01 uploaded processed features to S3; this project's equivalent is `utils/preprocessing.py`, which is not yet wired up to S3 — see the project skill notes), the confusion-matrix/feature-importance labels, and the quality gate threshold (0.85 ROC-AUC, matching the project proposal, instead of the tutor's 0.75 example).

In [ ]:
# After running this cell, restart the kernel before continuing if packages were upgraded.
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore

In [ ]:
# SageMaker MLflow App support.
%pip install --upgrade mlflow sagemaker-mlflow

## 0. Configuration

In [ ]:
import os
import json
from pathlib import Path

import boto3
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()

# ============================================================
# Student / team settings
# ============================================================
TEAM_ID = "team03"
STUDENT_ID = "s301"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "crypto-scam-detector"
REGION = "ap-southeast-1"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# ADAPTED: ROC-AUC quality gate matches the project proposal's stated threshold
# (Recall >= 0.85, Precision >= 0.75, F1 >= 0.80, ROC-AUC >= 0.85), not the tutor's
# 0.75 example value. Defined here so both the comparison plot and the gate check
# below stay in sync.
QUALITY_GATE = 0.85

# ============================================================
# MLflow App config
# ------------------------------------------------------------
# Preferred:
# 1. Team-level config copied/shared from Notebook 01A:
#       mlflow_app_config_team03.json
# 2. Student-specific config from Notebook 01A:
#       mlflow_app_config_team03_s301.json
# 3. Any local config matching this team:
#       mlflow_app_config_team03_*.json
#
# Important:
# Do not use another team's MLflow ARN. With team-level IAM
# restriction, wrong-team access should fail with 403.
# ============================================================

TEAM_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG_FILE = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

config_candidates = [
    TEAM_CONFIG_FILE,
    STUDENT_CONFIG_FILE,
    *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json")),
]

MLFLOW_APP_ARN = None
MLFLOW_EXPERIMENT = f"{COURSE}/{TEAM_ID}/Experiment1"
mlflow_app_config = {}
config_used = None

for config_file in config_candidates:
    if config_file.exists():
        mlflow_app_config = json.loads(config_file.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = mlflow_app_config.get("MLFLOW_APP_ARN")
        MLFLOW_EXPERIMENT = mlflow_app_config.get("EXPERIMENT_NAME", MLFLOW_EXPERIMENT)
        config_used = config_file
        break

# Fallback for classroom testing only.
# Team03's MLflow App ARN from Notebook 01A, used if the config file is not
# available in this Studio workspace.
DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-J5AYUG4AJHVW"
)

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print(
        "[WARNING] No local MLflow config file found. "
        "Using DEFAULT_MLFLOW_APP_ARN. Make sure this ARN belongs to your own team."
    )
else:
    print(f"Loaded MLflow App config from {config_used}")

# Validate config team, if present
config_team_id = mlflow_app_config.get("TEAM_ID") or mlflow_app_config.get("team_id")
if config_team_id and config_team_id != TEAM_ID:
    raise ValueError(
        f"Config file team mismatch: config TEAM_ID={config_team_id}, notebook TEAM_ID={TEAM_ID}. "
        "Do not use another team's MLflow config."
    )

REGISTERED_MODEL = f"iti113-{TEAM_ID}-{PROJECT_NAME}"
RANDOM_STATE = 42

print("=" * 70)
print("ITI113 Baseline Experiments — SageMaker MLflow App")
print("=" * 70)
print(f"Team               : {TEAM_ID}")
print(f"Student            : {STUDENT_ID}")
print(f"S3 Prefix          : {PREFIX}")
print(f"MLflow App ARN     : {MLFLOW_APP_ARN}")
print(f"MLflow Experiment  : {MLFLOW_EXPERIMENT}")
print(f"Registered Model   : {REGISTERED_MODEL}")
print(f"Quality gate (AUC) : {QUALITY_GATE}")
print(f"SageMaker Role     : {role}")
print("=" * 70)

## 1. Connect to SageMaker MLflow App

In [ ]:
import mlflow
import boto3

sm_client = boto3.client("sagemaker", region_name=REGION)

# ============================================================
# Safety check for team-level MLflow restriction
# ------------------------------------------------------------
# The selected MLflow App must have ResourceTag/TeamId = TEAM_ID.
# If a student accidentally uses another team's ARN, this should either:
# - fail with AccessDenied / 403 due to IAM restriction, or
# - fail this explicit validation before logging.
# ============================================================

try:
    tag_response = sm_client.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}
    print("MLflow App tags:")
    for k, v in mlflow_app_tags.items():
        print(f"  {k}: {v}")

    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App."
        )

    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")

except Exception as e:
    print("\n[ERROR] Could not validate MLflow App team tag.")
    print("This usually means one of the following:")
    print("1. The MLflow App ARN belongs to another team and IAM correctly blocked access.")
    print("2. The MLflow App is missing the TeamId tag.")
    print("3. The current role lacks permission to list tags for this MLflow App.")
    print(type(e).__name__, e)
    raise

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

COMMON_TAGS = {
    "course": COURSE,
    "semester": SEMESTER,
    "team_id": TEAM_ID,
    "student_id": STUDENT_ID,
    "team": TEAM_ID,          # kept for compatibility with previous notebooks
    "student": STUDENT_ID,    # kept for compatibility with previous notebooks
    "dataset": PROJECT_NAME,
    "tracking_backend": "sagemaker_mlflow_app",
    "mlflow_app_arn": MLFLOW_APP_ARN,
}

print(f"\nTracking URI : {mlflow.get_tracking_uri()}")

exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
print(f"Experiment   : {exp.name}")
print(f"Experiment ID: {exp.experiment_id}")

# Optional: generate a presigned UI URL if the current role has permission.
try:
    url_response = sm_client.create_presigned_mlflow_app_url(
        Arn=MLFLOW_APP_ARN
    )
    mlflow_ui_url = (
        url_response.get("AuthorizedUrl")
        or url_response.get("Url")
        or url_response.get("PresignedUrl")
    )
    print("\nOpen the SageMaker MLflow UI:")
    print(mlflow_ui_url)
except Exception as e:
    print("\nCould not generate MLflow UI URL from this role/session.")
    print(type(e).__name__, e)

## 2. Load Processed Data

**ADAPTED.** The tutor's version loads four already-vectorised CSVs (`train_features.csv`, `train_labels.csv`, `test_features.csv`, `test_labels.csv`) from S3 -- output of the tutor's Notebook 01 (EDA & data prep). This project's equivalent of that step is `utils/preprocessing.py` (cleans text, builds the engineered indicator features, splits train/test, fits TF-IDF), which currently writes locally rather than to S3 -- see the project skill notes for the planned S3 migration. Until then, this cell runs that same shared pipeline directly and builds `X_train`/`X_test` the same way the rest of the project (and the Streamlit app, at inference time) does: TF-IDF text features combined with the engineered indicator columns. `y_train`/`y_test` are binarised (1 = scam) here, the same way the tutor's own Notebook 01 binarised its target column, so the rest of this notebook's metric/plotting code (written for a 0/1 target) needs no further changes.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd, numpy as np
import scipy.sparse as sp

from utils.preprocessing import run_pipeline

SCAM_LABEL = "scam"  # update if the dataset uses a different positive-class label string

result = run_pipeline()
train_df, test_df, vectorizer = result["train_df"], result["test_df"], result["vectorizer"]

engineered_cols = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address", "wallet_address_count",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count", "message_length",
    "capital_letter_ratio", "has_numeric_content", "digit_count",
]

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

X_train = sp.hstack([X_train_tfidf, train_df[engineered_cols].values]).tocsr()
X_test = sp.hstack([X_test_tfidf, test_df[engineered_cols].values]).tocsr()

# Feature names for plots below (X_train is a sparse matrix, not a DataFrame,
# so there is no `.columns` the way the tutor's tabular version had).
FEATURE_NAMES = list(vectorizer.get_feature_names_out()) + engineered_cols

# Binarise labels (1 = scam), matching the tutor's own Notebook 01 target binarisation.
y_train = (train_df["label"] == SCAM_LABEL).astype(int)
y_test = (test_df["label"] == SCAM_LABEL).astype(int)

print(f'Train: {X_train.shape[0]} rows, {X_train.shape[1]} features')
print(f'Test : {X_test.shape[0]}  rows')
print(f'Scam rate — train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

## 3. Helper Functions

Reusable logging helpers so every run captures identical metrics and artifacts. Consistent logging is what makes the MLflow comparison view useful.

**ADAPTED:** confusion-matrix labels (`Legitimate`/`Scam` instead of the tutor's `No Disease`/`Disease`) and feature importance now reads names from `FEATURE_NAMES` instead of `X_train.columns` (sparse matrices don't have columns).

In [ ]:
import matplotlib.pyplot as plt, tempfile, mlflow.sklearn
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, ConfusionMatrixDisplay, RocCurveDisplay)

def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    metrics = {}
    for split, X, y in [('train',X_tr,y_tr),('test',X_te,y_te)]:
        yp = model.predict(X)
        yb = model.predict_proba(X)[:,1]
        metrics.update({
            f'{split}_accuracy' : round(accuracy_score(y,yp),4),
            f'{split}_f1'       : round(f1_score(y,yp),4),
            f'{split}_precision': round(precision_score(y,yp),4),
            f'{split}_recall'   : round(recall_score(y,yp),4),
            f'{split}_auc_roc'  : round(roc_auc_score(y,yb),4),
        })
    return metrics

def log_confusion_matrix(model, X_te, y_te, title):
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay.from_estimator(
        model, X_te, y_te,
        display_labels=['Legitimate','Scam'], cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as tmp:
        p = f'{tmp}/confusion_matrix.png'
        fig.savefig(p, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(p, artifact_path ='plots')
    plt.show(); plt.close()

def log_roc_curve(model, X_te, y_te, title):
    fig, ax = plt.subplots(figsize=(5,4))
    RocCurveDisplay.from_estimator(model, X_te, y_te, ax=ax)
    ax.plot([0,1],[0,1],'k--', label='Random baseline')
    ax.set_title(f'ROC Curve — {title}', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as tmp:
        p = f'{tmp}/roc_curve.png'
        fig.savefig(p, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(p, artifact_path ='plots')
    plt.show(); plt.close()

def log_feature_importance(model, feature_names, title, top_n=25):
    if not hasattr(model,'feature_importances_'): return
    imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)
    imp = imp.tail(top_n)  # ADAPTED: TF-IDF gives thousands of features, so only show the top ones
    fig, ax = plt.subplots(figsize=(6,7))
    imp.plot(kind='barh', ax=ax, color='#5B9BD5')
    ax.set_title(f'Feature Importance (top {top_n}) — {title}', fontweight='bold')
    plt.tight_layout()
    with tempfile.TemporaryDirectory() as tmp:
        p = f'{tmp}/feature_importance.png'
        fig.savefig(p, dpi=100, bbox_inches='tight')
        mlflow.log_artifact(p, artifact_path ='plots')
    plt.show(); plt.close()

print('Helper functions ready.')

## 4. Run 1 — Logistic Regression Baseline

The simplest model that makes sense for binary classification. Sets a performance floor that more complex models must beat.

In [ ]:
from sklearn.linear_model import LogisticRegression

run_name = f"{TEAM_ID}_{STUDENT_ID}_logistic_regression_baseline"

# ADAPTED: class_weight='balanced' added (kept from this project's previous baseline
# notebook) since the scam dataset is imbalanced -- the proposal prioritises recall.
LR_PARAMS = {'C':1.0, 'max_iter':1000, 'solver':'lbfgs', 'class_weight':'balanced', 'random_state':RANDOM_STATE}

with mlflow.start_run(run_name=run_name) as run:

    mlflow.set_tags(COMMON_TAGS)

    mlflow.log_params({**LR_PARAMS, 'model_type':'LogisticRegression',
                       'n_features':X_train.shape[1], 'train_size':X_train.shape[0]})

    lr = LogisticRegression(**LR_PARAMS)
    lr.fit(X_train, y_train)

    metrics = evaluate_model(lr, X_train, y_train, X_test, y_test)
    mlflow.log_metrics(metrics)
    log_confusion_matrix(lr, X_test, y_test, 'Logistic Regression')
    log_roc_curve(lr, X_test, y_test, 'Logistic Regression')
    mlflow.sklearn.log_model(lr, name ='model')
    LR_RUN_ID = run.info.run_id

print('LOGISTIC REGRESSION RESULTS')
for k,v in sorted(metrics.items()):
    print(f'  {"TRAIN" if k.startswith("train") else "TEST "} {k:<25}: {v}')
print(f'Run ID: {LR_RUN_ID}')

## 5. Run 2 — Random Forest Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier

run_name = f"{TEAM_ID}_{STUDENT_ID}_random_forest_baseline"

RF_PARAMS = {'n_estimators':100,'max_depth':6,'min_samples_leaf':4,
             'class_weight':'balanced','random_state':RANDOM_STATE,'n_jobs':-1}

with mlflow.start_run(run_name=run_name) as run:

    mlflow.set_tags(COMMON_TAGS)

    mlflow.log_params({**RF_PARAMS, 'model_type':'RandomForestClassifier',
                       'n_features':X_train.shape[1], 'train_size':X_train.shape[0]})

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)

    metrics = evaluate_model(rf, X_train, y_train, X_test, y_test)
    mlflow.log_metrics(metrics)
    log_confusion_matrix(rf, X_test, y_test, 'Random Forest')
    log_roc_curve(rf, X_test, y_test, 'Random Forest')
    log_feature_importance(rf, FEATURE_NAMES, 'Random Forest')
    mlflow.sklearn.log_model(rf, name ='model')
    RF_RUN_ID = run.info.run_id

print('RANDOM FOREST RESULTS')
for k,v in sorted(metrics.items()):
    print(f'  {"TRAIN" if k.startswith("train") else "TEST "} {k:<25}: {v}')

## 6. Run 3 — Hyperparameter Tuning with Nested Runs

Each hyperparameter combination is logged as a **child run** under a parent run. This keeps the MLflow UI organised — the parent summarises the best result.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

param_grid = [
    {'n_estimators': 50,  'max_depth': 4, 'min_samples_leaf': 2},
    {'n_estimators':100,  'max_depth': 4, 'min_samples_leaf': 4},
    {'n_estimators':100,  'max_depth': 6, 'min_samples_leaf': 2},
    {'n_estimators':200,  'max_depth': 6, 'min_samples_leaf': 4},
    {'n_estimators':200,  'max_depth': 8, 'min_samples_leaf': 2},
]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
best_auc, best_model, best_params = 0, None, None

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_rf_tuning_parent") as parent:

    mlflow.set_tags(COMMON_TAGS)

    mlflow.log_param('n_candidates', len(param_grid))
    mlflow.log_param('cv_folds', 5)

    for i, params in enumerate(param_grid):
        all_p = {**params, 'class_weight':'balanced', 'random_state':RANDOM_STATE}
        with mlflow.start_run(run_name=f'rf_candidate_{i+1:02d}', nested=True):
            mlflow.log_params(all_p)
            m = RandomForestClassifier(**all_p, n_jobs=-1)
            cv_scores = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
            mlflow.log_metric('cv_auc_mean', round(cv_scores.mean(),4))
            mlflow.log_metric('cv_auc_std',  round(cv_scores.std(),4))
            m.fit(X_train, y_train)
            test_m = evaluate_model(m, X_train, y_train, X_test, y_test)
            mlflow.log_metrics(test_m)
            #log model artifact for this candidate run
            mlflow.sklearn.log_model(m, name = "model")

            print(f'Candidate {i+1}: {params} | CV AUC={cv_scores.mean():.4f} | Test AUC={test_m["test_auc_roc"]}')
            if cv_scores.mean() > best_auc:
                best_auc, best_model, best_params = cv_scores.mean(), m, all_p

    mlflow.log_metric('best_cv_auc', round(best_auc,4))
    TUNING_RUN_ID = parent.info.run_id

print(f'\nBest CV AUC : {best_auc:.4f}')
print(f'Best params : {best_params}')

## 7. Compare Runs and Register Best Model

In [ ]:
from mlflow.tracking import MlflowClient
import mlflow

client = MlflowClient()

exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)

if exp is None:
    raise ValueError(f"Experiment not found: {MLFLOW_EXPERIMENT}")

runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.test_auc_roc DESC"]
)

if runs.empty:
    raise ValueError("No finished MLflow runs found yet.")

cols = [
    "tags.mlflow.runName",
    "metrics.test_auc_roc",
    "metrics.test_accuracy",
    "metrics.test_f1",
    "metrics.test_recall",
    "run_id"
]

avail = [c for c in cols if c in runs.columns]
print(runs[avail].head(8).to_string(index=False))

best_run = runs.iloc[0]
BEST_RUN_ID = best_run["run_id"]
BEST_AUC = best_run.get("metrics.test_auc_roc", 0)
BEST_RUN_NAME = best_run.get("tags.mlflow.runName", "unknown")

print(f"\nBest run: {BEST_RUN_NAME} | AUC={BEST_AUC:.4f}")

In [ ]:
# Visual comparison of top completed runs in the current experiment.
fig, ax = plt.subplots(figsize=(10,5))

metric_cols = [
    'metrics.test_auc_roc',
    'metrics.test_accuracy',
    'metrics.test_f1',
    'metrics.test_recall'
]
avail_m = [c for c in metric_cols if c in runs.columns]

# Use the top 3 finished runs by AUC.
top_runs = runs.head(3)

labels = [c.replace('metrics.test_','').upper() for c in avail_m]
x = np.arange(len(labels))
w = 0.25

for idx,(_, row) in enumerate(top_runs.iterrows()):
    vals = [row.get(m,0) for m in avail_m]
    ax.bar(
        x + idx*w,
        vals,
        w,
        label=row.get('tags.mlflow.runName', f'run{idx}')
    )

ax.set_xticks(x+w)
ax.set_xticklabels(labels)
ax.set_ylim(0,1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Test Set Metrics', fontweight='bold')
# ADAPTED: reference the QUALITY_GATE variable (0.85) instead of a hardcoded 0.75
ax.axhline(QUALITY_GATE, linestyle='--', alpha=0.5)
ax.text(len(labels)-0.5, QUALITY_GATE+0.01, f'AUC gate ({QUALITY_GATE})', fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Quality Gate and Best Model Selection
# ==========================================================

print("=" * 60)
print("QUALITY GATE")
print("=" * 60)

if BEST_AUC < QUALITY_GATE:

    print(f"Quality gate FAILED")
    print(f"Best AUC : {BEST_AUC:.4f}")
    print(f"Required : {QUALITY_GATE:.2f}")
    print("\nReview feature engineering or hyperparameters before proceeding.")

    BEST_MODEL_URI = None

else:

    print(f"Quality gate PASSED")
    print(f"Best AUC : {BEST_AUC:.4f}")

    # Export information for Notebook 03
    BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"

    print("\nBest MLflow Run")
    print("-" * 60)
    print(f"Run Name   : {BEST_RUN_NAME}")
    print(f"Run ID     : {BEST_RUN_ID}")
    print(f"Model URI  : {BEST_MODEL_URI}")
    print(f"Best AUC   : {BEST_AUC:.4f}")

print("=" * 60)

In [ ]:
# ==========================================================
# Notebook 03 Inputs
# ==========================================================

print("\nCopy these values into Notebook 03 if required:\n")

print(f'BEST_RUN_ID    = "{BEST_RUN_ID}"')

if BEST_MODEL_URI is not None:
    print(f'BEST_MODEL_URI = "{BEST_MODEL_URI}"')

print(f"BEST_AUC       = {BEST_AUC:.4f}")
print(f'BEST_RUN_NAME  = "{BEST_RUN_NAME}"')

In [ ]:
# Write to JSON file so Notebook 03 can read it.
import json

best_model_info = {
    "team_id": TEAM_ID,
    "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME,
    "best_run_id": BEST_RUN_ID,
    "best_run_name": BEST_RUN_NAME,
    "best_auc": float(BEST_AUC),
    "best_model_uri": BEST_MODEL_URI,
    "mlflow_tracking_uri": MLFLOW_APP_ARN,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "tracking_backend": "sagemaker_mlflow_app",
    "mlflow_app_team_tag": TEAM_ID,
}

with open("best_model.json", "w") as f:
    json.dump(best_model_info, f, indent=4)

print("Saved best model information to best_model.json")
print(json.dumps(best_model_info, indent=2))

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
artifacts = client.list_artifacts(BEST_RUN_ID)

for a in artifacts:
    print(a.path, a.is_dir)

In [ ]:
# Load best model directly from the best MLflow run and test inference
loaded = mlflow.sklearn.load_model(BEST_MODEL_URI)

# ADAPTED: X_test is a sparse matrix (no .head()); slice rows directly instead.
sample = X_test[:5]
preds = loaded.predict(sample)
probas = loaded.predict_proba(sample)[:, 1]

results = pd.DataFrame({
    "actual": y_test.head(5).values,
    "predicted": preds,
    "probability": probas.round(3),
    "correct": preds == y_test.head(5).values
})

print("Sample predictions from best MLflow run model:")
print(results.to_string(index=False))

In [ ]:
print("=" * 55)
print("NOTEBOOK 02 COMPLETE")
print("=" * 55)
print(f"Experiment : {MLFLOW_EXPERIMENT}")
print(f"MLflow App : {MLFLOW_APP_ARN}")
print(f"Best model : {BEST_RUN_NAME} | AUC={BEST_AUC:.4f}")
print(f"Best run ID: {BEST_RUN_ID}")
print(f"Model URI  : {BEST_MODEL_URI}")
print()
print("Next: Notebook 03 — run the SageMaker Pipeline using the same team S3 folder.")
print("For AWS-native tracking, use the same SageMaker MLflow App ARN in later notebooks.")

---
## Checklist before Notebook 03

- [ ] Notebook 01A created or reused the SageMaker MLflow App with TeamId tag
- [ ] `mlflow_app_config_team03.json` or `mlflow_app_config_team03_s301.json` is available, or `DEFAULT_MLFLOW_APP_ARN` was verified
- [ ] SageMaker MLflow App connection working
- [ ] MLflow App tag `TeamId` matches notebook `TEAM_ID`
- [ ] Logistic Regression baseline logged to SageMaker MLflow App
- [ ] Random Forest baseline logged to SageMaker MLflow App
- [ ] Hyperparameter tuning nested runs visible in the SageMaker MLflow UI
- [ ] Quality gate passed (Best Test AUC ≥ 0.85)
- [ ] Best MLflow run identified
- [ ] BEST_RUN_ID recorded
- [ ] BEST_MODEL_URI generated successfully
- [ ] Able to load the best model from the MLflow Run URI and make predictions
- [ ] `best_model.json` generated with MLflow App tracking details

→ Continue to Notebook 03 to orchestrate training, model registration and deployment using SageMaker Pipelines.

## Download model if needed

In [ ]:
# ============================================================
# Find MLflow model URI, load model, and download model artifact
# For SageMaker MLflow App / Serverless MLflow
# ============================================================

import os
import json
from pathlib import Path

import mlflow
import mlflow.sklearn


# ------------------------------------------------------------
# 1. Load best model information saved by Notebook 02
# ------------------------------------------------------------

BEST_MODEL_JSON = "best_model.json"

if not os.path.exists(BEST_MODEL_JSON):
    raise FileNotFoundError(
        f"{BEST_MODEL_JSON} not found. "
        "Run the model selection cell first."
    )

with open(BEST_MODEL_JSON, "r") as f:
    best_model_info = json.load(f)

print("Best model information:")
print(json.dumps(best_model_info, indent=2))


# ------------------------------------------------------------
# 2. Set MLflow tracking URI
# ------------------------------------------------------------

mlflow_tracking_uri = best_model_info["mlflow_tracking_uri"]
best_model_uri = best_model_info["best_model_uri"]
best_run_id = best_model_info["best_run_id"]

mlflow.set_tracking_uri(mlflow_tracking_uri)

print("\nMLflow tracking URI:")
print(mlflow_tracking_uri)

print("\nBest run ID:")
print(best_run_id)

print("\nBest model URI:")
print(best_model_uri)


# ------------------------------------------------------------
# 3. Load the model directly from MLflow
# ------------------------------------------------------------

loaded_model = mlflow.sklearn.load_model(best_model_uri)

print("\nLoaded model type:")
print(type(loaded_model))


# ------------------------------------------------------------
# 4. Download the MLflow model artifact folder locally
# ------------------------------------------------------------

download_dir = Path("downloaded_mlflow_models")
download_dir.mkdir(parents=True, exist_ok=True)

local_model_path = mlflow.artifacts.download_artifacts(
    artifact_uri=best_model_uri,
    dst_path=str(download_dir)
)

print("\nDownloaded model folder:")
print(local_model_path)


# ------------------------------------------------------------
# 5. List downloaded model files
# ------------------------------------------------------------

print("\nDownloaded files:")

for root, dirs, files in os.walk(local_model_path):
    level = root.replace(local_model_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        file_path = os.path.join(root, file)
        size_bytes = os.path.getsize(file_path)
        print(f"{indent}  {file}  ({size_bytes} bytes)")


# ------------------------------------------------------------
# 6. Optional quick prediction test
# ------------------------------------------------------------
# Uncomment if X_test exists in your notebook.
#
# sample = X_test[:5]
# predictions = loaded_model.predict(sample)
# print("\nSample predictions:")
# print(predictions)

---
## CryptoScam-AI addition: bridge to the local model registry (not in the tutor's notebook)

Notebook 03 (still to be built) will register the best model into **SageMaker's** Model Registry, which is meant to supersede this project's local `utils/model_registry.py` per the confirmed implementation plan. Until Notebook 03 exists, this cell keeps the Streamlit app working by saving `loaded_model` (the MLflow run selected above) as a new version in the local registry, exactly as the previous version of this notebook did.

In [ ]:
from utils.model_registry import save_model_version, promote_to_best

saved = save_model_version(
    loaded_model,
    metrics=best_model_info,
    notes=f"From SageMaker MLflow App run '{BEST_RUN_NAME}' ({BEST_RUN_ID}), AUC={BEST_AUC:.4f}. "
          "Bridge step -- superseded by SageMaker Model Registry once Notebook 03 is built.",
)

# Once reviewed and happy with it:
# promote_to_best(saved["version"])